# Class Balancing via Data Augmentation

**Goal**: Balance the unbalanced `training_data_praesentation.csv` by augmenting underrepresented classes to match the count of the most represented class.

**Techniques** (same as 03e):
- Back-translation (EN→DE→EN, EN→FR→EN)
- Contextual word substitution (RoBERTa-based)

**Quality Controls** (same as 03e):
- Length: 50-150% of original
- Vocabulary overlap ≥ 60%
- Semantic similarity ≥ 0.70 (sentence-transformers)

**Output**:
- `data/balanced_training_data.csv` with all original columns
- Augmented chunks marked as 'augmented' in Annotation column
- Statistics saved to `data/balancing_stats.json`


In [1]:
# Imports
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pandas as pd
import numpy as np
import torch
import json
import warnings
from tqdm.auto import tqdm
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer
import random
from collections import Counter

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
GPU: NVIDIA GeForce GTX 1060


## 1) Load Data and Analyze Class Distribution


In [2]:
# Load the unbalanced dataset
df = pd.read_csv('training_data_praesentation.csv')

print(f"Total samples: {len(df):,}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

# Handle NaN in frame_label (map to 'None' if needed)
df['frame_label'] = df['frame_label'].fillna('None')
if 'uneindeutig' in df['frame_label'].values:
    df['frame_label'] = df['frame_label'].replace({'uneindeutig': 'None'})

# Analyze class distribution
class_counts = df['frame_label'].value_counts().sort_index()
print(f"\n{'='*70}")
print("CLASS DISTRIBUTION (UNBALANCED)")
print(f"{'='*70}")
for label, count in class_counts.items():
    print(f"  {label:20s}: {count:4d} samples")
print(f"{'='*70}")

# Determine target count (max class count)
target_count = class_counts.max()
print(f"\n🎯 Target count per class: {target_count}")

# Calculate how many augmented samples needed per class
augmentation_needed = {}
for label, count in class_counts.items():
    needed = max(0, target_count - count)
    augmentation_needed[label] = needed
    if needed > 0:
        print(f"  {label:20s}: needs {needed:4d} augmented samples")
    else:
        print(f"  {label:20s}: already at target (no augmentation needed)")


Total samples: 2,000

Columns: ['chunk_id', 'chunk_text', 'dataset_split', 'frame_label', 'Annotation']

First few rows:
       chunk_id                                         chunk_text  \
0  chunk_000026  We know, of course, that without a U-turn from...   
1  chunk_000037  If, despite our warnings in today’s debate, th...   
2  chunk_000110  I certainly would, as I said earlier.\n\nAngel...   
3  chunk_000141  There is a body of opinion in the EU that want...   
4  chunk_000212  conserve fish stocks, and to help fishermen or...   

  dataset_split    frame_label Annotation  
0         train       Conflict   händisch  
1          test       Economic   händisch  
2         train    Moral Value   händisch  
3         train    Moral Value   händisch  
4          test  Powerlessness   händisch  

CLASS DISTRIBUTION (UNBALANCED)
  Conflict            :  439 samples
  Economic            :  457 samples
  Human Impact        :  234 samples
  Moral Value         :  248 samples
  None       

## 2) Load MarianMT Models for Back-Translation


In [3]:
print("Loading MarianMT models for back-translation...")
print("This may take a few minutes on first run.")

# EN→DE
print("\n  [1/4] Loading EN→DE...")
en_de_tokenizer = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-de')
en_de_model = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-de', use_safetensors=True).to(device)

# DE→EN
print("  [2/4] Loading DE→EN...")
de_en_tokenizer = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-de-en')
de_en_model = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-de-en', use_safetensors=True).to(device)

# EN→FR
print("  [3/4] Loading EN→FR...")
en_fr_tokenizer = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-fr')
en_fr_model = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-fr', use_safetensors=True).to(device)

# FR→EN
print("  [4/4] Loading FR→EN...")
fr_en_tokenizer = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-fr-en')
fr_en_model = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-fr-en', use_safetensors=True).to(device)

print("✅ MarianMT models loaded")


Loading MarianMT models for back-translation...
This may take a few minutes on first run.

  [1/4] Loading EN→DE...
  [2/4] Loading DE→EN...
  [3/4] Loading EN→FR...
  [4/4] Loading FR→EN...
✅ MarianMT models loaded


## 3) Load Sentence Similarity Model


In [4]:
print("Loading sentence similarity model...")
similarity_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Sentence similarity model loaded")


Loading sentence similarity model...
✅ Sentence similarity model loaded


## 4) Define Augmentation Functions (Same as 03e)


In [5]:
def translate_text(text, src_tokenizer, src_model, tgt_tokenizer, tgt_model, max_length=512):
    """Generic back-translation function"""
    try:
        inputs = src_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        translated = src_model.generate(**inputs, max_length=max_length, num_beams=4, early_stopping=True)
        intermediate = src_tokenizer.decode(translated[0], skip_special_tokens=True)

        inputs = tgt_tokenizer(intermediate, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        back = tgt_model.generate(**inputs, max_length=max_length, num_beams=4, early_stopping=True)
        result = tgt_tokenizer.decode(back[0], skip_special_tokens=True)
        return result
    except Exception as e:
        return None


def back_translate_german(text):
    return translate_text(text, en_de_tokenizer, en_de_model, de_en_tokenizer, de_en_model)


def back_translate_french(text):
    return translate_text(text, en_fr_tokenizer, en_fr_model, fr_en_tokenizer, fr_en_model)


def contextual_word_substitution(text, p=0.15):
    """RoBERTa-base contextual word substitution
    
    - Replace 10-20% of words (p=0.15 default)
    - Find top-5 similar tokens via cosine similarity
    - Keep only substitutions with similarity > 0.75
    - Preserve domain keywords
    """
    from transformers import RobertaTokenizer, RobertaModel
    import torch.nn.functional as F
    
    # Load RoBERTa-base once (cached as function attribute)
    if not hasattr(contextual_word_substitution, 'roberta_model'):
        print("  [Loading roberta-base for contextual substitution (one-time)...]")
        contextual_word_substitution.roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
        contextual_word_substitution.roberta_model = RobertaModel.from_pretrained('roberta-base', use_safetensors=True).to(device)
        contextual_word_substitution.roberta_model.eval()
        
        # Pre-compute vocabulary embeddings for efficient similarity search
        print("  [Pre-computing vocabulary embeddings...]")
        vocab_size = contextual_word_substitution.roberta_tokenizer.vocab_size
        vocab_embeddings = contextual_word_substitution.roberta_model.get_input_embeddings()
        with torch.no_grad():
            vocab_embs = vocab_embeddings.weight.cpu()  # [vocab_size, 768]
            # Normalize for cosine similarity
            vocab_embs = F.normalize(vocab_embs, p=2, dim=1)
        contextual_word_substitution.vocab_embeddings = vocab_embs
        print("  ✅ RoBERTa ready for contextual substitution")
    
    tokenizer = contextual_word_substitution.roberta_tokenizer
    model = contextual_word_substitution.roberta_model
    vocab_embeddings = contextual_word_substitution.vocab_embeddings
    
    # Protected domain keywords
    protected = {'EU', 'European', 'Union', 'Parliament', 'Commission', 'Brexit', 
                 'eurozone', 'euro', 'referendum', 'UK', 'Germany', 'France'}
    
    words = text.split()
    new_words = []
    
    for word in words:
        # Skip if: protected, non-alpha, or random skip (maintain ~85% of original text)
        if word in protected or not word.isalpha() or random.random() > p:
            new_words.append(word)
            continue
        
        try:
            # Get contextual embedding for the word
            # Context window: surrounding ±5 words
            idx = len(new_words)
            start = max(0, idx - 5)
            end = min(len(words), idx + 6)
            context = ' '.join(words[start:end])
            
            # Tokenize with RoBERTa
            inputs = tokenizer(context, return_tensors="pt", padding=True, truncation=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(**inputs)
                hidden_states = outputs.last_hidden_state[0]  # [seq_len, 768]
            
            # Find position of target word in tokenized sequence
            # Simple heuristic: find token closest to word position
            tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
            
            # Locate target word in tokenized sequence
            word_idx = None
            word_lower = word.lower()
            for idx, token in enumerate(tokens):
                if word_lower in token.lower().replace('Ġ', ''):
                    word_idx = idx
                    break
            
            if word_idx is None:
                # Fallback: use middle token
                word_idx = len(tokens) // 2
            
            # Get contextual embedding for the target word
            word_embedding = hidden_states[word_idx].cpu()  # [768]
            
            # Compute cosine similarity with all vocabulary tokens
            word_embedding_norm = F.normalize(word_embedding.unsqueeze(0), p=2, dim=1)  # [1, 768]
            similarities = torch.matmul(
                word_embedding_norm, 
                vocab_embeddings.T
            ).squeeze()  # [vocab_size]
            
            # Get top-5 similar tokens
            top_k = 6  # Get 6 to have 5 alternatives (exclude the word itself)
            top_indices = torch.topk(similarities, k=top_k).indices
            
            # Filter candidates: similarity > 0.75 and not the original word
            candidates = []
            for idx in top_indices:
                idx = idx.item()
                sim_score = similarities[idx].item()
                
                # Decode token (handle BPE)
                candidate = tokenizer.decode([idx]).strip()
                
                # Check: not original word, similarity > 0.75, is alphabetic, length > 2
                if (candidate.lower() != word.lower() and 
                    sim_score > 0.75 and 
                    candidate.isalpha() and 
                    len(candidate) > 2):
                    candidates.append((candidate, sim_score))
            
            # If we have valid candidates, pick randomly from top candidates
            if candidates:
                replacement, sim_score = random.choice(candidates[:3])  # Pick from top 3
                
                # Preserve original capitalization
                if word[0].isupper():
                    replacement = replacement.capitalize()
                
                new_words.append(replacement)
            else:
                # No valid replacement found
                new_words.append(word)
        
        except Exception as e:
            # Fallback: keep original on any error
            new_words.append(word)
    
    return ' '.join(new_words)


def quality_check(original, augmented):
    """Check augmented text quality (same as 03e)"""
    if augmented is None or len(augmented.strip()) == 0:
        return False, 'empty'
    
    # Length check: 50-150% of original
    orig_len = len(original.split())
    aug_len = len(augmented.split())
    if aug_len < 0.5 * orig_len or aug_len > 1.5 * orig_len:
        return False, 'length'
    
    # Vocabulary overlap ≥ 60%
    orig_vocab = set(original.lower().split())
    aug_vocab = set(augmented.lower().split())
    overlap = len(orig_vocab & aug_vocab) / len(orig_vocab) if len(orig_vocab) > 0 else 0
    if overlap < 0.60:
        return False, 'vocabulary'
    
    # Semantic similarity ≥ 0.70 using sentence-transformers
    try:
        orig_emb = similarity_model.encode(original)
        aug_emb = similarity_model.encode(augmented)
        similarity = np.dot(orig_emb, aug_emb) / (np.linalg.norm(orig_emb) * np.linalg.norm(aug_emb))
        if similarity < 0.70:
            return False, 'similarity'
    except:
        return False, 'similarity_error'
    
    return True, 'pass'

print("✅ Augmentation functions defined")


✅ Augmentation functions defined


## 5) Generate Augmented Data for Class Balancing


In [6]:
# Initialize tracking
augmented_data = []
augmentation_stats = {
    'total_original': len(df),
    'total_augmented': 0,
    'target_count': target_count,
    'by_class': {},
    'by_method': {'back_translate_german': 0, 'back_translate_french': 0, 'contextual_substitution': 0},
    'quality_rejections': {'empty': 0, 'length': 0, 'vocabulary': 0, 'similarity': 0, 'similarity_error': 0}
}

print(f"\n{'='*70}")
print("GENERATING AUGMENTED DATA")
print(f"{'='*70}\n")

# Process each class that needs augmentation
for label in sorted(augmentation_needed.keys()):
    needed = augmentation_needed[label]
    if needed == 0:
        print(f"{label}: No augmentation needed (already at target)")
        augmentation_stats['by_class'][label] = {
            'original': class_counts[label],
            'augmented': 0,
            'total': class_counts[label]
        }
        continue
    
    print(f"\n{label}: Need {needed} augmented samples")
    print("-" * 60)
    
    # Get all samples for this class
    label_df = df[df['frame_label'] == label].copy()
    
    # Define augmentation methods to cycle through
    methods = ['back_translate_german', 'back_translate_french', 'contextual_substitution']
    method_functions = {
        'back_translate_german': back_translate_german,
        'back_translate_french': back_translate_french,
        'contextual_substitution': contextual_word_substitution
    }
    
    # Generate augmented samples
    augmented_count = 0
    attempts = 0
    max_attempts = needed * 10  # Safety limit
    
    pbar = tqdm(total=needed, desc=f"{label}")
    
    while augmented_count < needed and attempts < max_attempts:
        # Randomly sample a row from this class
        row = label_df.sample(n=1).iloc[0]
        chunk_text = row['chunk_text']
        chunk_id = row['chunk_id']
        
        # Cycle through methods
        method = methods[attempts % len(methods)]
        aug_func = method_functions[method]
        
        # Generate augmented text
        try:
            aug_text = aug_func(chunk_text)
        except Exception as e:
            aug_text = None
        
        # Quality check
        passed, reason = quality_check(chunk_text, aug_text)
        
        if passed:
            # Create augmented row with all original columns
            aug_row = row.to_dict()
            aug_row['chunk_id'] = f"{chunk_id}_aug_{augmented_count}"
            aug_row['chunk_text'] = aug_text
            aug_row['Annotation'] = 'augmented'  # Mark as augmented
            
            augmented_data.append(aug_row)
            augmentation_stats['by_method'][method] += 1
            augmented_count += 1
            pbar.update(1)
        else:
            # Track rejection reason
            augmentation_stats['quality_rejections'][reason] += 1
        
        attempts += 1
    
    pbar.close()
    
    # Record stats for this class
    augmentation_stats['by_class'][label] = {
        'original': class_counts[label],
        'augmented': augmented_count,
        'total': class_counts[label] + augmented_count,
        'attempts': attempts
    }
    
    print(f"  ✅ Generated {augmented_count}/{needed} (attempts: {attempts})")

augmentation_stats['total_augmented'] = len(augmented_data)

print(f"\n{'='*70}")
print("✅ AUGMENTATION COMPLETE!")
print(f"{'='*70}")
print(f"Original samples: {augmentation_stats['total_original']:,}")
print(f"Augmented samples: {augmentation_stats['total_augmented']:,}")
print(f"Total samples: {augmentation_stats['total_original'] + augmentation_stats['total_augmented']:,}")
print(f"\nAugmentation methods used:")
for method, count in augmentation_stats['by_method'].items():
    print(f"  {method:25s}: {count:4d}")
print(f"\nQuality rejections:")
for reason, count in augmentation_stats['quality_rejections'].items():
    print(f"  {reason:25s}: {count:4d}")



GENERATING AUGMENTED DATA


Conflict: Need 18 augmented samples
------------------------------------------------------------


Conflict:   0%|          | 0/18 [00:00<?, ?it/s]

  [Loading roberta-base for contextual substitution (one-time)...]


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  [Pre-computing vocabulary embeddings...]
  ✅ RoBERTa ready for contextual substitution
  ✅ Generated 18/18 (attempts: 22)
Economic: No augmentation needed (already at target)

Human Impact: Need 223 augmented samples
------------------------------------------------------------


...

  ✅ Generated 209/209 (attempts: 258)

None: Need 152 augmented samples
------------------------------------------------------------


None:   0%|          | 0/152 [00:00<?, ?it/s]

  ✅ Generated 152/152 (attempts: 204)

Powerlessness: Need 140 augmented samples
------------------------------------------------------------


Powerlessness:   0%|          | 0/140 [00:00<?, ?it/s]

  ✅ Generated 140/140 (attempts: 177)

✅ AUGMENTATION COMPLETE!
Original samples: 2,000
Augmented samples: 742
Total samples: 2,742

Augmentation methods used:
  back_translate_german    :  264
  back_translate_french    :  165
  contextual_substitution  :  313

Quality rejections:
  empty                    :    0
  length                   :   26
  vocabulary               :  168
  similarity               :    5
  similarity_error         :    0


In [7]:
# Ensure Annotation column exists in original data
if 'Annotation' not in df.columns:
    df['Annotation'] = 'händisch'  # or whatever the original value should be

# Create augmented DataFrame
augmented_df = pd.DataFrame(augmented_data)

# Combine original + augmented
balanced_df = pd.concat([df, augmented_df], ignore_index=True)

print(f"\nBalanced dataset:")
print(f"  Total samples: {len(balanced_df):,}")
print(f"\n  Class distribution:")
balanced_counts = balanced_df['frame_label'].value_counts().sort_index()
for label, count in balanced_counts.items():
    original = class_counts[label]
    augmented = count - original
    print(f"    {label:20s}: {count:4d} ({original:4d} original + {augmented:4d} augmented)")

# Create data directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Save balanced dataset with explicit UTF-8 encoding
balanced_df.to_csv('data/balanced_training_data.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ Saved data/balanced_training_data.csv")

# Convert numpy/pandas int64 to native Python int for JSON serialization
def convert_to_serializable(obj):
    """Recursively convert numpy/pandas types to native Python types"""
    if isinstance(obj, dict):
        return {key: convert_to_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    else:
        return obj

# Save statistics (with type conversion)
augmentation_stats_serializable = convert_to_serializable(augmentation_stats)
with open('data/balancing_stats.json', 'w') as f:
    json.dump(augmentation_stats_serializable, f, indent=2)
print(f"✅ Saved data/balancing_stats.json")

print(f"\n{'='*70}")
print("🎉 CLASS BALANCING COMPLETE!")
print(f"{'='*70}")



Balanced dataset:
  Total samples: 2,742

  Class distribution:
    Conflict            :  457 ( 439 original +   18 augmented)
    Economic            :  457 ( 457 original +    0 augmented)
    Human Impact        :  457 ( 234 original +  223 augmented)
    Moral Value         :  457 ( 248 original +  209 augmented)
    None                :  457 ( 305 original +  152 augmented)
    Powerlessness       :  457 ( 317 original +  140 augmented)

✅ Saved data/balanced_training_data.csv
✅ Saved data/balancing_stats.json

🎉 CLASS BALANCING COMPLETE!


## 7) Verify Results


In [5]:
# Verify balanced distribution by reading the saved CSV file
import pandas as pd
import os

print("\nFinal Verification:")
print(f"{'='*70}")

# Load the balanced dataset from CSV
csv_path = 'data/balanced_training_data.csv'
if not os.path.exists(csv_path):
    print(f"❌ Error: {csv_path} not found!")
    print("   Please run the previous cells to generate the balanced dataset first.")
else:
    balanced_df = pd.read_csv(csv_path, keep_default_na=False)
    
    # Handle "None" as valid label (not null)
    if 'frame_label' in balanced_df.columns:
        balanced_df['frame_label'] = balanced_df['frame_label'].replace('', pd.NA)
    
    print(f"\n✅ Loaded balanced dataset from {csv_path}")
    print(f"   Total samples: {len(balanced_df):,}")
    
    # Check class balance
    balanced_counts = balanced_df['frame_label'].value_counts().sort_index()
    target_count = balanced_counts.max()  # Target is the maximum count
    
    print(f"\nClass counts (should all be equal to {target_count}):")
    all_balanced = True
    for label, count in balanced_counts.items():
        status = "✅" if count == target_count else "⚠️"
        if count != target_count:
            all_balanced = False
        print(f"  {status} {label:20s}: {count:4d} / {target_count}")
    
    if all_balanced:
        print(f"\n✅ All classes are balanced!")
    else:
        print(f"\n⚠️  Some classes are not balanced")
    
    # Check Annotation column
    print(f"\nAnnotation column distribution:")
    annotation_counts = balanced_df['Annotation'].value_counts()
    for annotation, count in annotation_counts.items():
        print(f"  {annotation:20s}: {count:4d}")
    
    # Check original vs augmented
    if 'Annotation' in balanced_df.columns:
        original_count = (balanced_df['Annotation'] != 'augmented').sum()
        augmented_count = (balanced_df['Annotation'] == 'augmented').sum()
        print(f"\n  Original samples: {original_count:,}")
        print(f"  Augmented samples: {augmented_count:,}")
    
    # Sample augmented data
    print(f"\n{'='*70}")
    print("Sample Augmented Chunks:")
    print(f"{'='*70}")
    augmented_df = balanced_df[balanced_df['Annotation'] == 'augmented']
    if len(augmented_df) > 0:
        augmented_samples = augmented_df.sample(min(3, len(augmented_df)))
        for idx, row in augmented_samples.iterrows():
            print(f"\n{row['frame_label']}:")
            print(f"  chunk_id: {row['chunk_id']}")
            print(f"  text: {row['chunk_text'][:150]}...")
            print(f"  annotation: {row['Annotation']}")
    else:
        print("  No augmented samples found in the dataset.")



Final Verification:

✅ Loaded balanced dataset from data/balanced_training_data.csv
   Total samples: 2,742

Class counts (should all be equal to 457):
  ✅ Conflict            :  457 / 457
  ✅ Economic            :  457 / 457
  ✅ Human Impact        :  457 / 457
  ✅ Moral Value         :  457 / 457
  ✅ None                :  457 / 457
  ✅ Powerlessness       :  457 / 457

✅ All classes are balanced!

Annotation column distribution:
  händisch            : 2000
  augmented           :  742

  Original samples: 2,000
  Augmented samples: 742

Sample Augmented Chunks:

Human Impact:
  chunk_id: chunk_200866_aug_102
  text: I was 11 years old at the time, and we have to move towards the 21st century, and we have to have a unity of goal, which means that we ultimately want...
  annotation: augmented

Human Impact:
  chunk_id: chunk_260609_aug_178
  text: I wish now to turn to the question of abortion, which other Members will want to raise as well. I have two specific questions for the Min